In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from scripts.plotting import *
from sklearn.preprocessing import StandardScaler
from scipy.linalg import svd
from scipy.spatial.distance import pdist, squareform, jaccard
from scipy.linalg import norm

In [ ]:
import scanpy as sc
import scvelo as scv

bdata = sc.read_h5ad("./data/pancreas/pancreas_inferred_velocity.h5ad")
scv.pl.velocity_embedding_stream(bdata, basis="umap", color="clusters", density=1.5, arrow_size=0.1)

In [ ]:
bdata

In [ ]:
X_2d = bdata.obsm["X_umap"]
cluster_labels = bdata.obs['clusters'].astype(str).values  # Ensure string type for mapping

# Extract cluster colors
cluster_colors = bdata.uns['clusters_colors']  # List of colors indexed by cluster order

# Get unique cluster names in the order stored in 'clusters_colors'
unique_clusters = np.unique(cluster_labels)

# Create a mapping from cluster names to colors
cluster_to_color = {cluster: color for cluster, color in zip(unique_clusters, cluster_colors)}

# Assign colors to each cell based on its cluster
cell_colors = [cluster_to_color[cluster] for cluster in cluster_labels]

plot_2d_quiver(X_2d, bdata.obsm["velocity_umap"], cell_colors, scale=0.2, cmap='coolwarm', 
               arrow_color='black', use_normalized=False)

In [ ]:
def scale_columns(X):
    return X / np.std(X, axis=0, keepdims=True)

In [ ]:
from scripts.plotting import *
from scripts.denoising import *
from scripts.TPS import *

X_2d = bdata.obsm["X_umap"]
X = bdata.layers["Ms"]
X = scale_columns(X)
tps = ThinPlateSpline(X_2d, n_control_points=2000)
tps.fit(X, dof_target=20)

jacobians = tps.compute_tps_jacobians(X_2d)
jacobians.shape

In [ ]:
def project_velocities(Y, jacobians):
    """
    Projects the velocities using least squares regression on the Jacobians.

    Parameters:
    - Y: np.ndarray of shape (num_cells, num_genes), the observed velocity matrix.
    - jacobians: list of np.ndarray, each of shape (num_genes, D), the Jacobian matrices.

    Returns:
    - projected_velocities: np.ndarray of shape (num_cells, D), the estimated velocity projections.
    - intercepts: np.ndarray of shape (num_cells,), the intercepts from the least squares fits.
    - residuals: np.ndarray of shape (num_cells, num_genes), the residuals after projection.
    """
    num_cells, num_genes = Y.shape
    D = jacobians[0].shape[1]  # Number of features in each Jacobian

    # Initialize output arrays
    projected_velocities = np.zeros((num_cells, D))
#     betas = np.zeros(num_cells)
    residuals = np.zeros_like(Y)

    for i in range(num_cells):
        # Augment the Jacobian with a column of ones to capture the intercept
        J = jacobians[i]  # Shape: (num_genes, D)
#         J_aug = np.hstack([J, np.ones((J.shape[0], 1))])  # Shape: (num_genes, D+1)

        # Solve for parameters using least squares
        beta, _, _, _ = np.linalg.lstsq(J, Y[i], rcond=None)

        # Separate out the slope coefficients and intercept
#         beta = params[:-1]  # Shape: (D,)
        projected_velocities[i] = beta
#         intercepts[i] = intercept

        # Compute residuals
        predicted = J @ beta
        residuals[i] = Y[i] - predicted

    return projected_velocities, residuals

Y = bdata.layers["velocity"]
Y = scale_columns(Y)
projected_velocities, residuals = project_velocities(Y, jacobians)

In [ ]:
plot_2d_quiver(X_2d, projected_velocities, cell_colors, scale=5, cmap='coolwarm', 
               arrow_color='black', use_normalized=False)

from scripts.steam_plot import *
tps_vf = ThinPlateSpline(X_2d, n_control_points=2000)
tps_vf.fit(projected_velocities, dof_target=20)

plot_velocity_streamplot(X_2d, tps_vf, cell_colors, 20)

In [ ]:
# Assuming jacobians has shape (N, d, D) and tps_vf.predict(X_2d) has shape (N, D)
vector_field = tps_vf.predict(X_2d)  # Shape: (N, D)
alignment_scores = []
N = X_2d.shape[0]
d = X.shape[1]

for i in range(jacobians.shape[1]):  # Iterate over Jacobian components
    jacobian_component = jacobians[:, i, :]  # Shape: (N, D)
    
    # Compute the dot product per row and sum
    numerator = np.sum(jacobian_component * vector_field, axis=1)
    
    # Compute normalization factor (L2 norms of each row, then sum over all rows)
    norm_factor = np.linalg.norm(jacobian_component, axis=1) * np.linalg.norm(vector_field, axis=1)
    
    # Avoid division by zero
    alignment_score = np.sum(np.abs(numerator / norm_factor)) / N
    alignment_scores.append(alignment_score)

In [ ]:
gene_names = bdata.var.index.tolist()
alignment_df = pd.DataFrame(data={"gene":gene_names,"gene_idx":np.arange(d),
                                  "alignment_score": alignment_scores})
alignment_df.sort_values("alignment_score")

In [ ]:
gene_expression_smoothed = tps.predict(X_2d)
plot_velocity_streamplot(X_2d, tps_vf, np.log(bdata.layers["spliced"].toarray()[:,1360]+1), 20)
plot_velocity_streamplot(X_2d, tps_vf, bdata.layers["Ms"][:,1360], 20)
plot_velocity_streamplot(X_2d, tps_vf, gene_expression_smoothed[:,1360], 20, title = "")

In [ ]:
plot_velocity_streamplot(X_2d, tps_vf, np.log(bdata.layers["spliced"].toarray()[:,1751]+1), 20)
plot_velocity_streamplot(X_2d, tps_vf, bdata.layers["Ms"][:,1751], 20)
plot_velocity_streamplot(X_2d, tps_vf, gene_expression_smoothed[:,1751], 20)

In [ ]:
import matplotlib.pyplot as plt

def plot_velocity_streamplot_with_gradient(X_2d, tps_vf, quiver_directions=None, 
                             scatter_color="blue", scatter_size=10, title=None):
    """
    Computes a velocity field on a grid, fills in the grid with predicted velocities,
    and plots a streamplot overlaying the original mesh grid points with customizable scatter color and size.
    Also overlays a quiver plot with user-provided quiver directions.

    Parameters
    ----------
    X_2d : np.ndarray
        A 2D numpy array of shape (n_points, 2) containing the original mesh grid points.
    tps_vf : object
        An object with a .predict() method that computes velocity given a point.
        Its predict() method should accept an array of shape (1, 2) and return an array of shape (1, 2).
    quiver_directions : np.ndarray or None, optional
        A 2D numpy array of shape (n_points, 2) specifying the quiver arrow directions for each X_2d point.
        If None, no quiver plot is added.
    scatter_color : str, optional
        Color of the scatter plot points (default is "blue").
    scatter_size : int or float, optional
        Size of the scatter plot points (default is 10).

    Returns
    -------
    None
    """
    # Compute the velocity on the grid
    X_grid = compute_velocity_on_grid(X_2d)

    # Extract unique x and y values from X_grid
    x_vals = np.unique(X_grid[:, 0])
    y_vals = np.unique(X_grid[:, 1])

    # Create a full mesh grid
    xx, yy = np.meshgrid(x_vals, y_vals)  # Shape: (ny, nx)

    # Create an empty velocity grid with NaN values
    ny, nx = xx.shape
    V_mesh = np.full((ny, nx, 2), np.nan)  # Holds the velocity field

    # Compute predicted velocity and store in V_mesh
    for point in X_grid:
        x_pt, y_pt = point[0], point[1]
        j_idx = np.where(x_vals == x_pt)[0]
        i_idx = np.where(y_vals == y_pt)[0]
        if j_idx.size > 0 and i_idx.size > 0:
            vel = tps_vf.predict(point.reshape(1, -1))[0]  # shape: (2,)
            V_mesh[i_idx[0], j_idx[0], :] = vel

    # Create the plot
    fig, ax = plt.subplots(figsize=(8, 6))

    # Scatter plot of original points
    ax.scatter(X_2d[:, 0], X_2d[:, 1], s=scatter_size, c=scatter_color, alpha=0.5)

    # Streamplot
    ax.streamplot(x_vals, y_vals, V_mesh[:, :, 0], V_mesh[:, :, 1], color='k', density=1, arrowsize=1.5)

    # Add quiver plot if quiver_directions is provided
    if quiver_directions is not None:
        ax.quiver(X_2d[:, 0], X_2d[:, 1], quiver_directions[:, 0], quiver_directions[:, 1], 
                  color="red", scale=25, width=0.003, headwidth=3, alpha=0.8)
    if title:
        ax.set_title(title)

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    plt.show()

plot_velocity_streamplot_with_gradient(X_2d, tps_vf, quiver_directions=jacobians[:,1873,:], 
                         scatter_color=gene_expression_smoothed[:,1873], scatter_size=10)
plot_velocity_streamplot_with_gradient(X_2d, tps_vf, quiver_directions=jacobians[:,1751,:], 
                         scatter_color=gene_expression_smoothed[:,1751], scatter_size=10)

In [ ]:
# Compute R^2 for each cell (column-wise calculation)
Y_mean_cell = np.mean(Y, axis=1)[:, np.newaxis]  # Mean velocity per cell
SS_total_cell = np.sum((Y - Y_mean_cell) ** 2, axis=1)  # Total variance per cell
SS_residual_cell = np.sum(residuals ** 2, axis=1)  # Sum of squared residuals per cell

R2_scores_cell = 1 - (SS_residual_cell / SS_total_cell)  # Compute R^2 for each cell
print(sum(R2_scores_cell>0) / len(R2_scores_cell))
# Plot histogram of R^2 values across cells
plt.figure(figsize=(8, 6))
plt.hist(R2_scores_cell, bins=50, edgecolor='black', alpha=0.75)
plt.xlabel("R^2 Score")
plt.ylabel("Number of Cells")
plt.title("Distribution of R^2 Scores Across Cells")
plt.show()

In [ ]:
# Compute R^2 for each gene
Y_mean = np.mean(Y, axis=0)  # Mean velocity for each gene
SS_total = np.sum((Y - Y_mean) ** 2, axis=0)  # Total variance
SS_residual = np.sum(residuals ** 2, axis=0)  # Sum of squared residuals

R2_scores_gene = 1 - (SS_residual / SS_total)  # Compute R^2 for each gene
R2_scores_gene_gt1 = R2_scores_gene[abs(R2_scores_gene) <= 1]
print(sum(R2_scores_gene>0) / len(R2_scores_gene))
# Plot histogram of R^2 values
plt.figure(figsize=(8, 6))
plt.hist(R2_scores_gene, bins=50, edgecolor='black', alpha=0.75)
plt.xlabel("R^2 Score")
plt.ylabel("Number of Genes")
plt.title("Distribution of R^2 Scores Across Genes")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_2d_quiver(X, vectors, color, scale=1, cmap='coolwarm', arrow_color='black', 
                    use_normalized=False, highlight_index=None):
    """
    Plots a 2D quiver (vector field) and highlights a specific cell's position and arrow.

    Args:
        X (np.ndarray): 2D coordinates of shape (n, 2).
        vectors (np.ndarray): Vector field of shape (n, 2).
        color (array-like): Colors for each cell.
        scale (float): Scaling factor for arrows.
        cmap (str): Colormap for arrow colors.
        arrow_color (str): Arrow edge color.
        use_normalized (bool): If True, normalizes vectors before plotting.
        highlight_index (int): Index of the cell to highlight.

    Returns:
        None (Displays plot).
    """
    # Normalize vectors if requested
    if use_normalized:
        norms = np.linalg.norm(vectors, axis=-1, keepdims=True)
        norms[norms == 0] = 1  # Prevent division by zero
        vectors = vectors / norms  # Normalize

    # Extract components
    X_positions, Y_positions = X[:, 0], X[:, 1]
    U_vectors, V_vectors = vectors[:, 0], vectors[:, 1]

    # Create figure
    plt.figure(figsize=(8, 6))
    plt.scatter(X[:, 0], X[:, 1], c=color, cmap=cmap, s=10, alpha=0.3)
    plt.quiver(X_positions, Y_positions, U_vectors, V_vectors, 
               color=arrow_color, angles='xy', scale_units='xy', scale=scale)

    # Highlight the specified cell and its quiver
    if highlight_index is not None:
        # Get the specific position and vector
        x_highlight, y_highlight = X[highlight_index, 0], X[highlight_index, 1]
        u_highlight, v_highlight = vectors[highlight_index, 0], vectors[highlight_index, 1]

        # Mark the cell position
        plt.scatter(x_highlight, y_highlight, 
                    color='red', s=40, edgecolors='black', label="Highlighted Cell")

        # Draw the highlighted quiver in red
        plt.quiver(x_highlight, y_highlight, u_highlight, v_highlight, 
                   color='red', angles='xy', scale_units='xy', scale=scale/10, linewidth=2)

    # Labels and title
    plt.xlabel("X Coordinate")
    plt.ylabel("Y Coordinate")
    plt.axis("equal")
    plt.legend()

    # Show plot
    plt.show()

In [ ]:
import numpy as np

def find_closest_row(matrix, vector):
    distances = np.linalg.norm(matrix - vector, axis=1)  # Compute Euclidean distances
    return np.argmin(distances)  # Return the index of the closest row


closest_index = find_closest_row(X_2d, [-10,4])

print(f"Closest row index: {closest_index}")

In [ ]:
import matplotlib.pyplot as plt

# Usage:
i = 2333
i = 571
plot_2d_quiver(X_2d, projected_velocities, cell_colors, scale=5, cmap='coolwarm', 
               arrow_color='black', use_normalized=True, highlight_index=i)

# Extract relevant data
jacobian_matrix = jacobians[i, :, :]  # Shape: (1945, 2)
velocity_vector = bdata.layers["velocity"][i, :]  # Shape: (1945,)

valid_indices = np.abs(velocity_vector) >= 0.1
filtered_jacobian = jacobian_matrix[valid_indices]
filtered_velocity = velocity_vector[valid_indices]

filtered_jacobian = jacobian_matrix[valid_indices]
filtered_velocity = velocity_vector[valid_indices]
filtered_jacobian[filtered_velocity < 0] *= -1
angles = np.arctan2(filtered_jacobian[:, 1], filtered_jacobian[:, 0])  # Angle in radians

plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)
ax.hist(angles, bins=50, alpha=0.7, edgecolor="black")

# Step 5: Add projected velocity vector
projected_vector = projected_velocities[i, :]  # Shape: (2,)
vector_angle = np.arctan2(projected_vector[1], projected_vector[0])
vector_length = np.linalg.norm(projected_vector)

# Plot the vector as a line
ax.plot([vector_angle, vector_angle], [0, 10], color='red', linewidth=2, label="Projected Velocity")

# Labels and title
ax.set_title("Radial Distribution of gradient angle", va='bottom')
ax.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LogisticRegression

i = 2333
i = 571
# Extract relevant data
jacobian_matrix = jacobians[i, :, :]  # Shape: (1945, 2)
velocity_vector = bdata.layers["velocity"][i, :]  # Shape: (1945,)


epsilon = 0.1  # Define margin threshold

# Assign labels with a margin
labels = np.where(velocity_vector > epsilon, 1, 
         np.where(velocity_vector < -epsilon, 0, -1))  # -1 indicates ambiguous region

# Remove ambiguous points from training
valid_indices = labels != -1  # Mask to keep only confident samples
jacobian_filtered = jacobian_matrix[valid_indices]
labels_filtered = labels[valid_indices]

# Fit logistic regression model without intercept
clf = LogisticRegression(fit_intercept=False)
clf.fit(jacobian_filtered, labels_filtered)

# Compute prediction accuracy
accuracy = clf.score(jacobian_filtered, labels_filtered)
print(f"Prediction Accuracy: {accuracy:.4f}")

# Compute decision boundary (solve w1*x + w2*y = 0 for y)
x_vals = np.linspace(jacobian_filtered[:, 0].min(), jacobian_filtered[:, 0].max(), 100)
w = clf.coef_[0]
y_vals = -(w[0] * x_vals) / w[1]  # Solve for y

# Assign colors based on velocity values
colors = np.where(velocity_vector < -epsilon, 'red', 
         np.where(velocity_vector > epsilon, 'blue', 'gray'))
alpha_values = np.where(colors == 'gray', 0, 0.4)

# Normalize velocity vectors to length 1
projected_vector = projected_velocities[i, :]
projected_vector_norm = np.linalg.norm(projected_vector)
projected_vector_normalized = 2 * projected_vector / projected_vector_norm

# Scatter plot of data points
plt.figure(figsize=(8, 6))
plt.scatter(jacobian_matrix[:, 0], jacobian_matrix[:, 1], c=colors, 
            alpha=alpha_values, edgecolors=None)

# Plot decision boundary as a line
plt.plot(x_vals, y_vals, 'k-', linewidth=2, label="Decision Boundary (No Intercept)")

# Plot velocity vectors
scale = 0.2  # Adjust the length of arrows for better visualization
plt.quiver(0, 0, 
           projected_vector_normalized[0], projected_vector_normalized[1], 
           angles='xy', scale_units='xy', scale=1/scale, width=0.01, color='red', alpha=1)

# Labels and title
plt.xlabel("Jacobian Component 1")
plt.ylabel("Jacobian Component 2")
plt.title("Logistic Regression Decision Boundary with Velocity Vectors")
plt.axhline(0, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)

# Add legend
legend_patches = [mpatches.Patch(color='blue', label='Positive Velocity'),
                  mpatches.Patch(color='red', label='Negative Velocity')]
plt.legend(handles=legend_patches)

# Show plot
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_gradient_boundary(gradient):
    """
    Plots a gradient vector and the separation boundary where df changes sign.

    Parameters:
    -----------
    gradient : np.ndarray
        A 2D numpy array of shape (2,) representing the gradient vector.
    """
    # Create a grid of points
    x_vals = np.linspace(-5, 5, 100)
    y_vals = np.linspace(-5, 5, 100)
    xx, yy = np.meshgrid(x_vals, y_vals)

    # Compute df = grad_x * x + grad_y * y
    df = gradient[0] * xx + gradient[1] * yy

    # Plot contour where df = 0 (decision boundary)
    contour = plt.contour(xx, yy, df, levels=[0], colors="red", linewidths=3, linestyles="dashed")

    # Plot gradient vector
    plt.quiver(0, 0, gradient[0], gradient[1], color="blue", scale=5, width=0.01, headwidth=4)

    # Labels for positive and negative df regions
    mid_x, mid_y = -gradient[1], gradient[0]  # Perpendicular direction to gradient
    plt.text(mid_x+1, mid_y, r"$dS > 0$", fontsize=18, color="black", ha="center", va="center", fontweight="bold")
    plt.text(-mid_x-1, -mid_y, r"$dS < 0$", fontsize=18, color="black", ha="center", va="center", fontweight="bold")

    # Formatting
    plt.axhline(0, color="black", linewidth=1)
    plt.axvline(0, color="black", linewidth=1)
    plt.xlim(-5, 5)
    plt.ylim(-5, 5)
    plt.xlabel("x", fontsize=16, fontweight="bold")
    plt.ylabel("y", fontsize=16, fontweight="bold")
    plt.title("Gradient Vector and Separation Boundary", fontsize=20, fontweight="bold")
    plt.grid(True, linestyle="--", linewidth=0.8)

    plt.show()

# Example usage with gradient [2, -1]
plot_gradient_boundary(np.array([2, -1]))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from sklearn.linear_model import LogisticRegression

epsilon = 0.05  # Set a fixed threshold for filtering
num_samples, num_genes = jacobians.shape[:2]  # Number of samples & genes

# Initialize tracking arrays
correct_counts = np.zeros(num_genes)  # Count of correct predictions per gene
incorrect_counts = np.zeros(num_genes)  # Count of incorrect predictions per gene
accuracies = []
correct_gene_matrix = np.zeros((num_samples, num_genes), dtype=int)  # Store per-cell results

# Iterate over i
for i in range(num_samples):
    jacobian_matrix = jacobians[i, :, :]  # Shape: (num_genes, 2)
    velocity_vector = bdata.layers["velocity"][i, :]  # Shape: (num_genes,)

    # Apply epsilon filtering
    pos_indices = velocity_vector >= epsilon
    neg_indices = velocity_vector <= -epsilon

    # Assign labels based on thresholding
    labels = np.full(num_genes, -1)  # Initialize with -1 (ambiguous)
    labels[pos_indices] = 1  # Positive class
    labels[neg_indices] = 0  # Negative class

    # Remove ambiguous points
    valid_indices = labels != -1
    jacobian_filtered = jacobian_matrix[valid_indices]
    labels_filtered = labels[valid_indices]

    if len(labels_filtered) == 0:  # Skip if no valid training data
        accuracies.append(None)
        continue

    # Compute positive label ratio for threshold adjustment
    num_pos = np.sum(labels_filtered == 1)
    num_neg = np.sum(labels_filtered == 0)
    threshold = num_neg / (num_pos + num_neg) if (num_pos + num_neg) > 0 else 0.5

    # Fit logistic regression
    clf = LogisticRegression(fit_intercept=False, class_weight="balanced", penalty="l2")
    clf.fit(jacobian_filtered, labels_filtered)

    # Predict probabilities instead of direct classification
    predicted_probs = clf.predict_proba(jacobian_filtered)[:, 1]  # Probability of class 1
    predicted_labels = (predicted_probs >= threshold).astype(int)  # Adjusted threshold

    # Compute accuracy
    accuracy = np.mean(predicted_labels == labels_filtered)
    accuracies.append(accuracy)

    # Get correctly predicted indices
    correct_predictions = (predicted_labels == labels_filtered)
    correct_gene_indices_filtered = np.where(correct_predictions)[0]

    # Map back to original indices
    original_indices = np.where((labels == 0) | (labels == 1))[0]  # Indices before filtering
    correct_gene_indices = original_indices[correct_gene_indices_filtered]

    # Update per-gene correctness matrix
    correct_gene_matrix[i, correct_gene_indices] = 1  # Mark correct predictions

    # Update counts
    correct_counts[correct_gene_indices] += 1  # Increment correct count
    incorrect_counts[original_indices] += 1  # Increment total attempts
    incorrect_counts[correct_gene_indices] -= 1  # Remove correct from incorrect count

In [ ]:
# Compute proportion of correct predictions per gene
gene_accuracy = correct_counts / (correct_counts + incorrect_counts)
gene_accuracy = np.nan_to_num(gene_accuracy)  # Handle divisions by zero

# Plot histogram of gene-wise accuracy
plt.figure(figsize=(8, 5))
plt.hist(gene_accuracy, bins=20, edgecolor='black', alpha=0.75)
plt.xlabel("Gene Accuracy (Proportion Correct)")
plt.ylabel("Number of Genes")
plt.title("Histogram of Gene-Wise Prediction Accuracy")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Compute total occurrences of each gene in training (correct + incorrect)
total_occurrences = correct_counts + incorrect_counts

# Plot histogram of gene occurrences
plt.figure(figsize=(8, 5))
plt.hist(total_occurrences, bins=20, edgecolor='black', alpha=0.75)
plt.xlabel("Total Occurrences in Classification")
plt.ylabel("Number of Genes")
plt.title("Histogram of Gene Occurrences in Classification")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# Remove None values from accuracies for plotting
valid_accuracies = [acc for acc in accuracies if acc is not None]
# Plot histogram
plt.figure(figsize=(8, 5))
plt.hist(valid_accuracies, bins=20, edgecolor='black', alpha=0.75)
plt.xlabel("Accuracy")
plt.ylabel("Frequency")
plt.title("Histogram of Logistic Regression Accuracy")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import numpy as np

def project_velocities_weighted(Y, jacobians, gene_weight):
    """
    Projects the velocities using weighted least squares regression on the Jacobians.

    Parameters:
    - Y: np.ndarray of shape (num_cells, num_genes), the observed velocity matrix.
    - jacobians: list of np.ndarray, each of shape (num_genes, D), the Jacobian matrices.
    - gene_weights: np.ndarray of shape (num_genes,), weight for each gene based on classification accuracy.

    Returns:
    - projected_velocities: np.ndarray of shape (num_cells, D), the estimated velocity projections.
    - residuals: np.ndarray of shape (num_cells, num_genes), the residuals after projection.
    """
    num_cells, num_genes = Y.shape
    D = jacobians[0].shape[1]  # Number of features in each Jacobian

    # Initialize output arrays
    projected_velocities = np.zeros((num_cells, D))
    residuals = np.zeros_like(Y)

    for i in range(num_cells):
        # Extract Jacobian and velocity vector for cell i
        J = jacobians[i]  # Shape: (num_genes, D)
        v = Y[i]  # Shape: (num_genes,)

        # Create diagonal weight matrix W
        W = np.diag(gene_weights)  # Shape: (num_genes, num_genes)

        # Solve weighted least squares: (J^T W J) beta = J^T W v
        JTWJ = J.T @ W @ J
        JTWv = J.T @ W @ v

        # Solve for beta (use np.linalg.solve for numerical stability)
        beta = np.linalg.solve(JTWJ, JTWv)

        # Store projected velocities
        projected_velocities[i] = beta

        # Compute residuals
        predicted = J @ beta
        residuals[i] = v - predicted

    return projected_velocities, residuals

In [ ]:
# Apply weighted regression
Y = bdata.layers["velocity"]
Y = scale_columns(Y)

total_counts = correct_counts + incorrect_counts
gene_weights = (correct_counts+1) / (incorrect_counts+1)
gene_weights = (correct_counts+10) / (total_counts+10)
gene_weights = gene_weights = np.where(gene_weights > 0.5, gene_weights, 0)

projected_velocities, residuals = project_velocities_weighted(Y, jacobians, gene_weights)

In [ ]:
plot_2d_quiver(X_2d, projected_velocities, cell_colors, scale=5, cmap='coolwarm', 
               arrow_color='black', use_normalized=False)

from scripts.steam_plot import *
tps_vf = ThinPlateSpline(X_2d, n_control_points=2000)
tps_vf.fit(projected_velocities, dof_target=20)

plot_velocity_streamplot(X_2d, tps_vf, cell_colors, 20)

In [ ]:
# Compute R^2 for each cell (column-wise calculation)
Y_mean_cell = np.mean(Y, axis=1)[:, np.newaxis]  # Mean velocity per cell
SS_total_cell = np.sum((Y - Y_mean_cell) ** 2, axis=1)  # Total variance per cell
SS_residual_cell = np.sum(residuals ** 2, axis=1)  # Sum of squared residuals per cell

R2_scores_cell = 1 - (SS_residual_cell / SS_total_cell)  # Compute R^2 for each cell
print(sum(R2_scores_cell>0) / len(R2_scores_cell))
# Plot histogram of R^2 values across cells
plt.figure(figsize=(8, 6))
plt.hist(R2_scores_cell, bins=50, edgecolor='black', alpha=0.75)
plt.xlabel("R^2 Score")
plt.ylabel("Number of Cells")
plt.title("Distribution of R^2 Scores Across Cells")
plt.show()

In [ ]:
# Compute R^2 for each gene
Y_mean = np.mean(Y, axis=0)  # Mean velocity for each gene
SS_total = np.sum((Y - Y_mean) ** 2, axis=0)  # Total variance
SS_residual = np.sum(residuals ** 2, axis=0)  # Sum of squared residuals

R2_scores_gene = 1 - (SS_residual / SS_total)  # Compute R^2 for each gene
R2_scores_gene_gt1 = R2_scores_gene[abs(R2_scores_gene) <= 1]
print(sum(R2_scores_gene>0) / len(R2_scores_gene))
# Plot histogram of R^2 values
plt.figure(figsize=(8, 6))
plt.hist(R2_scores_gene, bins=50, edgecolor='black', alpha=0.75)
plt.xlabel("R^2 Score")
plt.ylabel("Number of Genes")
plt.title("Distribution of R^2 Scores Across Genes")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_residuals_vs_expression(Y, residuals, num_genes=5):
    """
    Plots residuals against gene expression for a few randomly selected genes.

    Parameters:
    - Y: np.ndarray of shape (num_cells, num_genes), the observed velocity matrix (gene expression).
    - residuals: np.ndarray of shape (num_cells, num_genes), the residuals after projection.
    - num_genes: int, number of genes to randomly select for plotting.
    """
    num_cells, total_genes = Y.shape
    selected_genes = np.random.choice(total_genes, num_genes, replace=False)  # Randomly select genes

    plt.figure(figsize=(8, 5))
    
    for gene in selected_genes:
        plt.scatter(Y[:, gene], residuals[:, gene], label=f"Gene {gene}", alpha=0.6, edgecolor="black", s=20)

    plt.xlabel("Gene Expression (Y)", fontsize=14, fontweight="bold")
    plt.ylabel("Residual Value", fontsize=14, fontweight="bold")
    plt.title("Residuals vs. Gene Expression", fontsize=16, fontweight="bold")
    plt.axhline(0, color="black", linestyle="--", linewidth=1)  # Reference line at 0
    plt.legend()
    plt.grid(True, linestyle="--", linewidth=0.5)

    plt.show()

# Example usage
plot_residuals_vs_expression(Y, residuals, num_genes=5)


In [ ]:
# Scatter plot of residual variance vs. gene weight
plt.figure(figsize=(6, 5))
plt.scatter(gene_weights, R2_scores_gene, color="tab:blue", alpha=0.6, edgecolor="black")

# Formatting
plt.xlabel("Gene Weight", fontsize=14, fontweight="bold")
plt.ylabel("R2 of gene", fontsize=14, fontweight="bold")
plt.title("R2 of gene vs. Gene Weight", fontsize=16, fontweight="bold")
# plt.xscale("log")  # Log scale for better visualization
plt.yscale("log")  # Log scale since variances can vary widely
plt.grid(True, linestyle="--", linewidth=0.6)

plt.show()